# Model 1 — GTZAN Genre CNN  🎛️

Trains the mel-spectrogram CNN defined in `app/vibe_model.py` (`GenreCNN`).
The app uses this model two ways:

* **genre / vibe tags** for each track, and
* the penultimate **64-d embedding** as a *vibe similarity* vector that the
  planner uses to order the set so neighbouring tracks *sound* alike.

> **Training is optional.** Without weights the app falls back to a strong
> hand-crafted DSP embedding — it still works, this just makes vibe-similarity
> smarter. The trained checkpoint is written to
> `models_weights/model_1_genre_cnn.pt` and auto-loaded on next server start.

## 1 · Dataset — GTZAN Genre Collection

1000 × 30 s clips, 10 genres (100 each).

* Official page: http://marsyas.info/downloads/datasets.html
* Convenient mirror (Kaggle): `andradaolteanu/gtzan-dataset-music-genre-classification`

Unpack so the layout is:
```
datasets/gtzan/genres_original/<genre>/<genre>.000xx.wav
```
(10 folders: blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock).

*Note: a handful of GTZAN files are known to be corrupt (e.g. `jazz.00054.wav`);
the loader below skips unreadable files automatically.*

In [1]:
import os, sys, glob, random, time
import numpy as np
# make the project package importable from train_models/
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path: sys.path.insert(0, ROOT)
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import librosa
from app.vibe_model import GenreCNN, log_mel, GTZAN_GENRES, WEIGHTS_PATH, EMBED_DIM
from app.config import ANALYSIS_SR, N_MELS, MODELS_DIR
device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| classes:', GTZAN_GENRES)

device: mps | classes: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


In [2]:
# ---- config ----
DATA_DIR = os.path.join(ROOT, 'datasets', 'gtzan', 'genres_original')
SR = ANALYSIS_SR
CLIP_SECONDS = 3.0          # each training example is a 3 s window
FRAMES = 128                # mel frames per window (matches vibe_model)
BATCH = 32; EPOCHS = 200; LR = 1e-3; VAL_FRAC = 0.15; SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert os.path.isdir(DATA_DIR), f'GTZAN not found at {DATA_DIR} — see the markdown above.'

In [3]:
# ---- index the audio files ----
files, labels = [], []
for gi, g in enumerate(GTZAN_GENRES):
    for fp in glob.glob(os.path.join(DATA_DIR, g, '*.wav')):
        files.append(fp); labels.append(gi)
print(f'found {len(files)} clips')
idx = np.arange(len(files)); np.random.shuffle(idx)
n_val = int(len(idx) * VAL_FRAC)
val_idx, train_idx = set(idx[:n_val].tolist()), set(idx[n_val:].tolist())

found 1000 clips


In [4]:
# ---- dataset: decode -> log-mel -> random 3 s window ----
class GTZAN(Dataset):
    def __init__(self, indices, train=True):
        self.items = [(files[i], labels[i]) for i in indices]
        self.train = train; self.cache = {}
    def __len__(self): return len(self.items)
    def _mel(self, fp):
        if fp in self.cache: return self.cache[fp]
        try:
            y, _ = librosa.load(fp, sr=SR, mono=True)
        except Exception:
            return None
        m = log_mel(y, SR)                 # (N_MELS, T)
        self.cache[fp] = m; return m
    def __getitem__(self, k):
        fp, lab = self.items[k]
        m = self._mel(fp)
        if m is None:  # skip corrupt file by returning a random other item
            return self.__getitem__((k + 1) % len(self.items))
        T = m.shape[1]
        s = random.randint(0, max(0, T - FRAMES)) if self.train else max(0, (T - FRAMES) // 2)
        w = m[:, s:s + FRAMES]
        if w.shape[1] < FRAMES:
            w = np.pad(w, ((0, 0), (0, FRAMES - w.shape[1])), mode='edge')
        return torch.from_numpy(w).float(), lab

train_ds = GTZAN(sorted(train_idx), True)
val_ds   = GTZAN(sorted(val_idx), False)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)
val_dl   = DataLoader(val_ds, batch_size=BATCH, num_workers=0)
print('train', len(train_ds), 'val', len(val_ds))

train 850 val 150


In [5]:
# ---- normalisation stats over a sample of the training mels ----
sample = []
for i in list(train_idx)[:120]:
    m = train_ds._mel(files[i])
    if m is not None: sample.append(m)
allm = np.concatenate(sample, axis=1)
MEAN, STD = float(allm.mean()), float(allm.std() + 1e-6)
print('mel mean/std', round(MEAN,2), round(STD,2))

mel mean/std -47.59 17.45


In [6]:
# ---- model / optim ----
model = GenreCNN(n_classes=len(GTZAN_GENRES)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
crit = nn.CrossEntropyLoss()
def norm(x): return (x - MEAN) / STD
print(sum(p.numel() for p in model.parameters()), 'parameters')

249514 parameters


In [7]:
# ---- train ----
best = 0.0
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tot = 0; correct = 0; loss_sum = 0
    for x, y in train_dl:
        x = norm(x).unsqueeze(1).to(device); y = y.to(device)
        opt.zero_grad()
        logits, _ = model(x)
        loss = crit(logits, y); loss.backward(); opt.step()
        loss_sum += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item(); tot += len(y)
    sched.step()
    # validation
    model.eval(); vc = vt = 0
    with torch.no_grad():
        for x, y in val_dl:
            x = norm(x).unsqueeze(1).to(device); y = y.to(device)
            logits, _ = model(x)
            vc += (logits.argmax(1) == y).sum().item(); vt += len(y)
    va = vc / max(vt, 1)
    print(f'epoch {ep+1:02d}  loss {loss_sum/tot:.3f}  train_acc {correct/tot:.3f}  val_acc {va:.3f}  ({time.time()-t0:.1f}s)')
    if va > best:
        best = va
        torch.save({'state_dict': model.state_dict(), 'genres': GTZAN_GENRES,
                    'norm': {'mean': MEAN, 'std': STD}, 'val_acc': va},
                   WEIGHTS_PATH)
print('best val acc', round(best,3), '->', WEIGHTS_PATH)

/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_14955/2754256127.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


epoch 01  loss 2.053  train_acc 0.294  val_acc 0.140  (12.3s)


/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_14955/2754256127.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


epoch 02  loss 1.807  train_acc 0.355  val_acc 0.273  (0.7s)


/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_14955/2754256127.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


...

epoch 198  loss 0.183  train_acc 0.942  val_acc 0.820  (0.7s)


/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_14955/2754256127.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


epoch 199  loss 0.164  train_acc 0.956  val_acc 0.813  (0.7s)


/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_14955/2754256127.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


epoch 200  loss 0.174  train_acc 0.946  val_acc 0.800  (0.7s)
best val acc 0.84 -> /Users/sam/Desktop/aiolearn/My_projects/AI_DJ/models_weights/model_1_genre_cnn.pt


## 3 · Done

The best checkpoint is saved to `models_weights/model_1_genre_cnn.pt`.
Restart the AI-DJ server; `app/vibe_model.py` detects the file and switches
from the DSP fallback to the learned embedding automatically.

Typical windowed val-accuracy on GTZAN with this compact net is ~0.75–0.82.
For the mix engine the *embedding geometry* matters more than raw accuracy —
even a moderately trained model gives smoother vibe-based ordering.